# CE49X: Introduction to Computational Thinking and Data Science for Civil Engineers
## Week 9: Decision Trees & Random Forests

**Instructor:** Dr. Eyuphan Koc
**Department of Civil Engineering, Bogazici University**
**Semester:** Spring 2026

Based on *Python Data Science Handbook* by Jake VanderPlas
Chapter 5: Machine Learning (Section 5.08 - Decision Trees and Random Forests)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, validation_curve, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, r2_score)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_blobs, make_moons, make_regression

%matplotlib inline

np.random.seed(49)
plt.rcParams['figure.dpi'] = 100

## Quick Recap: Where Are We?

Over the last two weeks we met two very different classifiers:

- **Naive Bayes** (Week 7) — a *generative* model that learns what each class "looks like" by fitting probability distributions.
- **Support Vector Machines** (Week 8) — a *discriminative* model that finds the widest possible *gap* between classes, optionally bent into curves with kernels.

Today we meet a third family: **Decision Trees** (and their ensemble cousin, **Random Forests**). Trees are also discriminative, but they are fundamentally different in spirit — instead of geometry, they ask **a sequence of yes/no questions**.

**Why should you care about trees?**
- They are **interpretable**: you can literally read the rules a tree learned. An inspector can audit them.
- They handle **mixed data types** (numeric + categorical) natively — no scaling, no one-hot explosion.
- They capture **non-linear interactions** for free, with almost no tuning.
- **Random Forests** are the default first-thing-to-try on any tabular civil-engineering dataset.
- Real applications: post-disaster damage classification, soil/material classification, equipment-failure prediction, and the kind of tabular project data you'll be working with in Weeks 10-12.

## Table of Contents

1. [Introduction to Decision Trees](#1.-Introduction-to-Decision-Trees)
2. [How Trees Learn: Splits, Gini, and a Tiny Example](#2.-How-Trees-Learn:-Splits,-Gini,-and-a-Tiny-Example)
3. [Overfitting and Tree Pruning](#3.-Overfitting-and-Tree-Pruning)
4. [From One Tree to a Forest: Bagging & Random Forests](#4.-From-One-Tree-to-a-Forest:-Bagging-&-Random-Forests)
5. [Application: Predicting Nepal Earthquake Damage](#5.-Application:-Predicting-Nepal-Earthquake-Damage)
6. [Regression, Summary & Practice](#6.-Regression,-Summary-&-Practice)
---

## 1. Introduction to Decision Trees

### What Is a Decision Tree?

A decision tree is, quite literally, a **flowchart of yes/no questions** that ends in a prediction. Starting at the top (the **root**), the tree asks one question, follows the branch corresponding to the answer, asks another question, and continues until it reaches a **leaf** that gives the final answer.

```
                 Is foundation cracked?
                 /                     \
              Yes                       No
              /                          \
     Is age > 50 yrs?              Is roof type heavy?
       /         \                   /            \
   UNSAFE       INSPECT          INSPECT          SAFE
```

**Key terms:**
- **Root node** — the first question (top of the tree)
- **Internal node** — any question along the way
- **Leaf** — a terminal prediction (no more questions)
- **Depth** — how many questions deep the tree can go

> **Key Idea**
> At each node, the tree picks the question that best separates the classes — the one that makes the resulting groups as "pure" as possible.

### Think Like a Tree: An Engineering Inspector

Imagine a structural inspector evaluating buildings after a moderate earthquake. They walk up to a building and ask, in sequence:

1. *"Is there visible foundation cracking?"* If **yes**, that's a strong signal of damage.
2. *"How old is the building?"* Older buildings with cracking are usually worse.
3. *"What's the superstructure made of?"* Mud-mortar-stone is very different from reinforced concrete.

Each question splits the buildings they've seen into smaller, more uniform groups. **That's exactly how a decision tree learns** — except it does it automatically by trying every possible question and picking the most informative one at each step.

### [LIVE] A Tree You Already Know

Before any code, here's a tree you make every morning without thinking about it:

```
                Is it raining?
                /            \
             Yes              No
             /                  \
   Take umbrella       Is rain forecast > 50%?
                            /          \
                         Yes            No
                         /                \
                Take umbrella        Leave it home
```

Notice three things:
1. Each split asks about **one feature** at a time.
2. The path from root to leaf is a **sequence of if-then rules** you can read out loud.
3. Different inputs land in different leaves — that's how the tree makes different predictions.

This is the entire conceptual machinery. The rest is just figuring out *which questions to ask*.

### Trees vs. SVM vs. Naive Bayes

| Property | Naive Bayes | SVM | **Decision Tree / RF** |
|---|---|---|---|
| Decision style | Probabilistic (Bayes' rule) | Geometric (margin) | **Sequential rules (if-then)** |
| Output | Class probability | Class label (+ distance) | **Class label / probability** |
| Needs scaling? | No | **Yes** (always) | **No** |
| Handles categorical features? | Limited | One-hot only | **Natively** |
| Captures non-linear interactions? | Limited | With kernel | **For free** |
| Interpretable? | Moderate | **Hard** (especially with RBF) | **Yes** (single tree) |

The headline: trees are the **interpretable, low-friction** option. RFs trade some of the interpretability of a single tree for much better accuracy.

> **Example: Civil Engineering**
> Trees are a natural fit for CE data because:
> - **Categorical features are everywhere** (foundation type, soil class, roof material) — trees handle them without one-hot encoding.
> - **No feature scaling** is needed — you can mix `age_years` (0-200) and `area_m2` (0-1000) directly.
> - **The if-then rules are auditable** — an engineer can read a tree and check whether its logic makes physical sense, which matters when models inform safety decisions.

---

## 2. How Trees Learn: Splits, Gini, and a Tiny Example

A tree learns by **greedily** picking the best question at each node. To do that, we need a way to measure *how mixed up* a group of samples is. The cleaner the groups after a split, the better the question.

### Gini Impurity: Measuring How "Mixed" a Group Is

Imagine reaching into a bag of marbles twice and pulling one out each time (with replacement). **Gini impurity** is the probability that the two marbles have *different colors*.

- Bag is all red → you'll never get a mismatch → Gini = **0** (pure).
- Bag is half red, half blue → mismatch happens half the time → Gini = **0.5** (max impurity for two classes).

Formally, for a node with class proportions $p_k$:

$$\text{Gini} = 1 - \sum_{k} p_k^2$$

**That's the only formula you need today.** When a tree picks a split, it chooses the question that drops the *weighted average* Gini of the two child nodes the most. (Sklearn also offers entropy, $-\sum p_k \log p_k$, but in practice the two almost never disagree on what to split.)

In [ ]:
def gini(counts):
    """Gini impurity of a node, given class counts."""
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)

# Three example groups
print(f"  10 good /  0 poor   ->  Gini = {gini([10, 0]):.3f}   (pure)")
print(f"   7 good /  3 poor   ->  Gini = {gini([7, 3]):.3f}   (mixed)")
print(f"   5 good /  5 poor   ->  Gini = {gini([5, 5]):.3f}   (max impurity for 2 classes)")

In [ ]:
# Visualize Gini for a binary class as we vary p
p = np.linspace(0, 1, 200)
gini_curve = 2 * p * (1 - p)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(p, gini_curve, color='steelblue', linewidth=2.5)
ax.fill_between(p, 0, gini_curve, color='steelblue', alpha=0.15)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.6)
ax.annotate('PURE\n(Gini = 0)', xy=(0, 0), xytext=(0.05, 0.08),
            fontsize=11, color='darkgreen')
ax.annotate('PURE\n(Gini = 0)', xy=(1, 0), xytext=(0.83, 0.08),
            fontsize=11, color='darkgreen')
ax.annotate('MAX IMPURITY\n(Gini = 0.5)', xy=(0.5, 0.5), xytext=(0.32, 0.40),
            fontsize=11, color='indianred',
            arrowprops=dict(arrowstyle='->', color='indianred', alpha=0.7))
ax.set_xlabel('Fraction of class 1  ($p$)', fontsize=12)
ax.set_ylabel('Gini impurity', fontsize=12)
ax.set_title('Gini impurity for a binary node', fontsize=13)
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3)
plt.show()

### The Greedy Split Algorithm

Here is the entire training procedure in pseudo-code:

```
function build_tree(samples):
    if samples are pure (or stop criterion hit):
        return Leaf(majority class)
    best_split = None
    best_gain  = 0
    for each feature f:
        for each candidate threshold t in samples[f]:
            left, right = split samples on (f <= t)
            gain = Gini(samples) - weighted_avg(Gini(left), Gini(right))
            if gain > best_gain:
                best_gain  = gain
                best_split = (f, t)
    left_tree  = build_tree(samples where best_split is true)
    right_tree = build_tree(samples where best_split is false)
    return Node(best_split, left_tree, right_tree)
```

It's **greedy** — at each node it picks the best split *right now*, never reconsidering. This is fast, but it's why trees can be sub-optimal on globally tricky problems.

### [TOGETHER] Hand-Built Split on a Tiny Beam Dataset

Let's make this concrete. Six steel beams have been inspected — each has a measured crack length and an overall condition rating. We'll find the best `crack_mm` threshold by hand.

In [ ]:
beams = pd.DataFrame({
    'beam_id':   [1, 2, 3, 4, 5, 6],
    'crack_mm':  [0.5, 1.2, 2.8, 5.1, 7.4, 9.0],
    'condition': ['good', 'good', 'good', 'poor', 'poor', 'poor'],
})
print(beams)

# Try every "midpoint between adjacent crack lengths" as a candidate threshold
sorted_cracks = np.sort(beams['crack_mm'].unique())
candidates = (sorted_cracks[:-1] + sorted_cracks[1:]) / 2

print("\nCandidate splits on crack_mm:")
print(f"{'threshold':>10} {'left':>14} {'right':>14} {'weighted Gini':>16}")
for t in candidates:
    left  = beams.loc[beams['crack_mm'] <= t, 'condition'].value_counts().reindex(['good','poor'], fill_value=0)
    right = beams.loc[beams['crack_mm'] >  t, 'condition'].value_counts().reindex(['good','poor'], fill_value=0)
    n_l, n_r = left.sum(), right.sum()
    weighted = (n_l * gini(left) + n_r * gini(right)) / (n_l + n_r)
    print(f"{t:>10.2f} {str(left.values):>14} {str(right.values):>14} {weighted:>16.3f}")

The split at **`crack_mm <= 3.95`** drives the weighted Gini all the way to **0** — it perfectly separates the good beams from the poor ones. No further questions needed; the tree stops with two pure leaves.

Now imagine doing this for 30 features and 50,000 buildings. That's exactly what `DecisionTreeClassifier` does in milliseconds.

### Decision Trees in Scikit-Learn

The API is the same `fit`/`predict` pattern from Naive Bayes and SVM:

```python
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(max_depth=5, random_state=49)
tree.fit(X_train, y_train)
preds = tree.predict(X_test)
```

A few parameters worth knowing now:
- `max_depth` — limits how many questions deep the tree can go (the main overfitting knob).
- `criterion` — `'gini'` (default) or `'entropy'`; rarely matters in practice.
- `random_state` — fixes tie-breaking when multiple splits look equally good.

In [ ]:
# A 4-class toy problem
X, y = make_blobs(n_samples=300, centers=4, cluster_std=1.2, random_state=49)

tree = DecisionTreeClassifier(random_state=49)  # no depth limit -> grows fully
tree.fit(X, y)

print(f"Tree depth   : {tree.get_depth()}")
print(f"Number of leaves: {tree.get_n_leaves()}")
print(f"Train accuracy : {tree.score(X, y):.3f}")

In [ ]:
def visualize_tree_decision(model, X, y, ax=None, cmap='viridis', title=''):
    """Plot the 2D decision regions of any sklearn classifier with .predict."""
    if ax is None:
        ax = plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=30, edgecolors='k', linewidths=0.5)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    return ax

fig, ax = plt.subplots(figsize=(8, 6))
visualize_tree_decision(tree, X, y, ax=ax,
                        title=f'Decision Tree (depth={tree.get_depth()}, {tree.get_n_leaves()} leaves)')
plt.show()

> **Key Insight: Axis-Aligned Boundaries**
> Trees can only split on *one feature at a time*, so the decision boundary is always made of horizontal and vertical pieces (hyper-rectangles in higher dimensions). Compare this to last week's SVM, which can draw smooth diagonals and curves with kernels. Each model has a "natural shape," and trees naturally carve the plane into rectangles.

In [ ]:
# Inspect the top of the tree: read the actual rules it learned
fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(tree, max_depth=3, filled=True, rounded=True,
          feature_names=['Feature 1', 'Feature 2'],
          class_names=[f'Class {i}' for i in range(4)],
          ax=ax)
ax.set_title('Top 3 levels of the fitted tree', fontsize=13)
plt.show()

Read the boxes top-down: the root asks a single question about Feature 1 or Feature 2, and the colors at the leaves show which class dominates. The Gini and `samples` numbers in each node tell you exactly how the impurity drops as you go deeper. **This is the readability superpower of trees — every prediction is a chain of if-then statements you can show to an engineer.**

---

## 3. Overfitting and Tree Pruning

If we leave a tree unrestricted, it will keep splitting until **every leaf is pure** — meaning it has *memorized* the training set. That's a recipe for terrible generalization.

### The Bias-Variance Trade-off, Tree Edition

| Tree depth | Behavior | Bias | Variance | Generalization |
|---|---|---|---|---|
| Very shallow (e.g. `max_depth=1`) | Asks only one question | High | Low | **Underfits** |
| Very deep / unrestricted | Memorizes every training point | Low | High | **Overfits** |
| Sweet spot | Captures real patterns, ignores noise | Balanced | Balanced | **Generalizes** |

This is the same story we saw with Ridge/Lasso back in Week 7 — the model complexity dial. For trees, the dial is mostly **`max_depth`** and **`min_samples_leaf`**.

### [LIVE] What Overfitting Looks Like

Let's grow trees of depth 1, 3, 7, and 15 on the *same* noisy data and watch the boundaries get crazier.

In [ ]:
X_moons, y_moons = make_moons(n_samples=400, noise=0.35, random_state=49)

depths = [1, 3, 7, 15]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, d in zip(axes.flat, depths):
    t = DecisionTreeClassifier(max_depth=d, random_state=49).fit(X_moons, y_moons)
    visualize_tree_decision(t, X_moons, y_moons, ax=ax, cmap='coolwarm',
                            title=f'max_depth = {d}   |   train acc = {t.score(X_moons, y_moons):.2%}')
plt.tight_layout()
plt.show()

Notice the **tiny islands** of color around stray points in the depth-15 plot. Those are the tree carving out leaves to memorize individual noisy samples. They look impressive on the training set (high accuracy!) but will be wrong for any new building that happens to land in one of those islands.

### The Validation-Curve Story (the "U-Curve")

The classic way to see overfitting is to plot **train accuracy** and **cross-validated accuracy** as we sweep a hyperparameter. The train curve climbs steadily; the validation curve rises, plateaus, then falls. The peak of the validation curve is your sweet spot.

In [ ]:
def plot_validation_curve(estimator, X, y, param_name, param_range,
                          cv=5, scoring='accuracy', ax=None):
    """Sweep one hyperparameter; plot mean train & CV scores with std bands."""
    train_scores, val_scores = validation_curve(
        estimator, X, y, param_name=param_name, param_range=param_range,
        cv=cv, scoring=scoring, n_jobs=-1)
    train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
    val_mean,   val_std   = val_scores.mean(axis=1),   val_scores.std(axis=1)

    if ax is None:
        ax = plt.gca()
    ax.plot(param_range, train_mean, color='steelblue', label='Training', linewidth=2)
    ax.fill_between(param_range, train_mean - train_std, train_mean + train_std,
                    color='steelblue', alpha=0.15)
    ax.plot(param_range, val_mean, color='indianred', label='Cross-validation', linewidth=2)
    ax.fill_between(param_range, val_mean - val_std, val_mean + val_std,
                    color='indianred', alpha=0.15)
    best_idx = int(np.argmax(val_mean))
    ax.axvline(param_range[best_idx], color='gray', linestyle='--', alpha=0.7)
    ax.set_xlabel(param_name)
    ax.set_ylabel(scoring)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    return param_range[best_idx], val_mean[best_idx]

depths = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(10, 6))
best_d, best_score = plot_validation_curve(
    DecisionTreeClassifier(random_state=49),
    X_moons, y_moons,
    param_name='max_depth', param_range=depths, ax=ax)
ax.annotate(f'best depth ≈ {best_d}', xy=(best_d, best_score),
            xytext=(best_d + 2, best_score - 0.05),
            fontsize=11, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray', alpha=0.7))
ax.text(15, 0.62, 'OVERFITTING', fontsize=12, color='indianred', fontweight='bold')
ax.text(1.2, 0.62, 'UNDERFITTING', fontsize=12, color='steelblue', fontweight='bold')
ax.set_title('The Bias-Variance Trade-off in Decision Trees', fontsize=13)
plt.show()
print(f"Best max_depth = {best_d} (CV accuracy = {best_score:.3f})")

### Pruning Knobs You'll Actually Use

In rough order of how often you'll touch them:

| Parameter | What it does | Typical first guess |
|---|---|---|
| `max_depth` | Hard cap on tree depth. Most direct overfitting control. | 5-15 |
| `min_samples_leaf` | A leaf must have at least this many samples. Smooths predictions. | 5-50 |
| `min_samples_split` | An internal node won't split unless it has at least this many samples. | 10-100 |
| `max_leaf_nodes` | Hard cap on total number of leaves. | (rarely used) |
| `ccp_alpha` | Cost-complexity pruning strength (post-pruning). | (rarely used; tune with grid search if you do) |

**Practical tip:** start by tuning `max_depth` and `min_samples_leaf` together. They're usually enough.

> **Key Insight: Civil Engineering Analogy**
> Pruning a tree is like the safety-factor `C` you tuned for SVM last week: too lax (deep tree, low `C`) and the model contorts to fit every training noise; too strict (depth = 1, huge `C`) and it's too crude to capture real structure. The U-curve is just the bias-variance trade-off, drawn in pictures.

---

## 4. From One Tree to a Forest: Bagging & Random Forests

A single tree is **high-variance** — small changes to the training data can produce a very different tree. The fix is wonderfully simple: **train many trees on slightly different views of the data and let them vote**.

### Bagging: Bootstrap Aggregation

**Bagging** = **B**ootstrap **AGG**regat**ING**. The recipe:

1. Take a **bootstrap sample** of the training data — sample $n$ rows *with replacement* from the original $n$ rows. About 37% of the original rows will be missing from any given bootstrap (those become the "out-of-bag" samples that we can use as a free validation set).
2. Train one tree on that bootstrap.
3. Repeat $T$ times to get $T$ trees.
4. **Predict** by majority vote (classification) or by averaging (regression).

> **Key Insight: Wisdom of Crowds**
> The average of many noisy, *independent* estimators is more accurate than any single one. The classic Galton story: 800 fairgoers estimated the weight of an ox; no single person was great, but the *median* guess was within 1% of the truth. Bagging is the same trick, applied to trees.

### What Makes a Random Forest "Random"

A bag of full-grown trees alone isn't enough — the trees end up looking too similar (they all latch onto the same dominant feature at the root) and averaging similar models doesn't help much. **Random Forests add a second source of randomness:**

> **At each split, only consider a random subset of features** (default: $\sqrt{p}$ features for classification, $p/3$ for regression).

This forces different trees to explore different splits, **decorrelating** them. Now averaging actually pays off.

So: Random Forest = **bagging + random feature subsetting**. Two simple tricks; massive accuracy improvement.

### [LIVE] Watching Trees Disagree, Then Agree

Let's fit forests of size 1, 5, and 100 on the same noisy moons data and watch the boundary smooth out.

In [ ]:
sizes = [1, 5, 100]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, n in zip(axes, sizes):
    rf = RandomForestClassifier(n_estimators=n, random_state=49, n_jobs=-1).fit(X_moons, y_moons)
    visualize_tree_decision(rf, X_moons, y_moons, ax=ax, cmap='coolwarm',
                            title=f'Random Forest, n_estimators = {n}\ntrain acc = {rf.score(X_moons, y_moons):.2%}')
plt.tight_layout()
plt.show()

From left to right: a single tree carves the plane into a few jagged regions; 5 trees average into something messier-looking but more reasonable; 100 trees produce a smooth (still axis-aligned underneath) boundary that follows the genuine curvature of the data.

### Random Forests in Scikit-Learn

```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,        # number of trees in the forest
    max_depth=None,          # let individual trees grow deep — bagging tames variance
    max_features='sqrt',     # the random-feature trick
    n_jobs=-1,               # train trees in parallel (use all cores)
    random_state=49,
)
rf.fit(X_train, y_train)
```

The defaults are unusually good. `n_estimators=200` is a sensible production starting point.

In [ ]:
# Quick fit on the 4-blob data from §2
X4, y4 = make_blobs(n_samples=300, centers=4, cluster_std=1.2, random_state=49)
X4_tr, X4_te, y4_tr, y4_te = train_test_split(X4, y4, test_size=0.3, random_state=49)

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=49)
rf.fit(X4_tr, y4_tr)
print(f"Train accuracy : {rf.score(X4_tr, y4_tr):.3f}")
print(f"Test  accuracy : {rf.score(X4_te, y4_te):.3f}")
print(f"\nIndividual trees can still be deep — bagging keeps the *ensemble* well-behaved:")
depths = [t.get_depth() for t in rf.estimators_]
print(f"  per-tree depth: min={min(depths)}, mean={np.mean(depths):.1f}, max={max(depths)}")

| Property | Single Decision Tree | Random Forest |
|---|---|---|
| Training time | Very fast | Slower (proportional to `n_estimators`) |
| Accuracy on tabular data | OK | **Usually much better** |
| Interpretability of single prediction | **Excellent** (read the rules) | Moderate (vote, but no single rule) |
| Feature importance | Noisy | **Reliable, averaged across trees** |
| Tendency to overfit | High without pruning | Low (bagging is the regularizer) |
| Tuning effort | `max_depth`, `min_samples_leaf` | Mostly works out of the box |

Bridge to §5: now let's apply both to a real, messy, civil-engineering dataset and use the feature-importance plot to actually *learn something* about earthquake damage.

---

## 5. Application: Predicting Nepal Earthquake Damage

**April 25, 2015 — 11:56 a.m. local time.** A magnitude 7.8 earthquake struck near Gorkha, Nepal, about 80 km northwest of Kathmandu. Roughly **9,000 people died**, ~22,000 were injured, and an estimated **600,000 buildings were destroyed or damaged**. It was the deadliest earthquake to hit Nepal since 1934.

In the months that followed, surveyors from the National Society for Earthquake Technology — Nepal (NSET-Nepal), Kathmandu Living Labs, and the Central Bureau of Statistics walked through affected districts and recorded the structural details and damage state of **762,106 buildings** — one of the largest post-disaster building-damage datasets ever assembled.

We'll work with a **50,000-row stratified subsample** of that dataset. The question we want to answer:

> *Given a building's location, age, and construction details, can we predict the damage grade it will suffer?*

If we can — and if we can identify *which features* drive the prediction — we have an empirical retrofit-prioritization tool. That's directly relevant for Bogazici students: the **North Anatolian Fault** runs ~20 km from this campus, and the Marmara region's building stock has a similar age and material distribution to rural Nepal.

### The Data

| Group | Columns | Notes |
|---|---|---|
| Identifier | `building_id` | Drop before modeling |
| Location | `geo_level_1_id`, `geo_level_2_id`, `geo_level_3_id` | Coarse → fine geographic regions |
| Geometry | `count_floors_pre_eq`, `age`, `area_percentage`, `height_percentage` | Numeric |
| Construction | `foundation_type`, `roof_type`, `ground_floor_type`, `other_floor_type`, `position`, `plan_configuration`, `land_surface_condition`, `legal_ownership_status` | Categorical strings |
| Materials | 11 binary `has_superstructure_*` flags | adobe_mud, mud_mortar_stone, rc_engineered, … |
| Use | `count_families`, 11 binary `has_secondary_use_*` flags | |
| **Target** | **`damage_grade`** ∈ {1, 2, 3} | 1 = low, 2 = medium, 3 = near-total destruction |

In [ ]:
df = pd.read_csv('data/nepal_buildings_sample.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Damage-grade distribution:")
print(df['damage_grade'].value_counts(normalize=True).sort_index().round(3))

**Class imbalance alert:** grade 2 dominates (~57%), grade 1 is rare (~10%). Plain accuracy will be misleading — a model that predicts "grade 2" for everything would already get 57% right. We'll watch **per-class recall** in the confusion matrix.

In [ ]:
# [TOGETHER] Three quick EDA views
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) class counts
df['damage_grade'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'gray', 'indianred'])
axes[0].set_title('Damage grade counts')
axes[0].set_xlabel('damage_grade')
axes[0].set_ylabel('buildings')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# (b) age distribution by damage grade (clip extreme ages for readability)
for grade, color in zip([1, 2, 3], ['steelblue', 'gray', 'indianred']):
    axes[1].hist(df.loc[df['damage_grade'] == grade, 'age'].clip(upper=100),
                 bins=25, alpha=0.5, label=f'grade {grade}', color=color)
axes[1].set_title('Age distribution by damage grade')
axes[1].set_xlabel('age (years)')
axes[1].set_ylabel('buildings')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# (c) damage rate by foundation type
foundation_damage = (df.groupby('foundation_type')['damage_grade']
                       .apply(lambda s: (s == 3).mean())
                       .sort_values())
foundation_damage.plot(kind='barh', ax=axes[2], color='indianred')
axes[2].set_title('P(damage_grade = 3) by foundation_type')
axes[2].set_xlabel('fraction destroyed')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### Preprocessing: Trees Don't Need Much, but They Need This

Trees are unusually low-maintenance:

- **No scaling required** — they look at thresholds, not distances. (Big contrast with SVM!)
- **No one-hot encoding required** — we use `OrdinalEncoder` to map each categorical level to an integer; trees can split on those integers directly without inflating dimensionality.

We'll use `ColumnTransformer` to apply ordinal encoding to the categorical columns and pass numeric columns through untouched.

In [ ]:
y = df['damage_grade']
X = df.drop(columns=['building_id', 'damage_grade'])

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()
print(f"{len(cat_cols)} categorical columns, {len(num_cols)} numeric columns")

preprocess = ColumnTransformer(
    transformers=[('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
                                         unknown_value=-1), cat_cols)],
    remainder='passthrough')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=49)
print(f"Train: {X_tr.shape},  Test: {X_te.shape}")

### Baseline: A Single Decision Tree

In [ ]:
tree_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=49)),
])
tree_pipe.fit(X_tr, y_tr)
print(f"Single tree  | train accuracy: {tree_pipe.score(X_tr, y_tr):.3f}")
print(f"Single tree  | test  accuracy: {tree_pipe.score(X_te, y_te):.3f}")

A noticeable train-test gap is normal for a single tree — it's our overfitting-evidence baseline. Now let's see how much a forest improves things.

### A Random Forest Should Do Better

In [ ]:
rf_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(n_estimators=200, max_features='sqrt',
                                   n_jobs=-1, random_state=49)),
])
rf_pipe.fit(X_tr, y_tr)
print(f"Random Forest | train accuracy: {rf_pipe.score(X_tr, y_tr):.3f}")
print(f"Random Forest | test  accuracy: {rf_pipe.score(X_te, y_te):.3f}")

print("\nClassification report (per-class precision / recall / F1):")
print(classification_report(y_te, rf_pipe.predict(X_te),
                            target_names=['grade 1 (low)', 'grade 2 (medium)', 'grade 3 (high)']))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_estimator(
    rf_pipe, X_te, y_te,
    display_labels=['grade 1', 'grade 2', 'grade 3'],
    cmap='Blues', normalize='true', ax=ax, colorbar=False)
ax.set_title('Random Forest — confusion matrix (row-normalized)')
plt.show()

### [TOGETHER] Feature Importance: What Predicts Damage?

Every split in every tree drops Gini by some amount. Sum those drops *per feature* across all trees in the forest, normalize, and you get a **feature importance** score: a number that tells you how much each feature contributed to reducing impurity.

> **Key Insight: This Is Why You Use Random Forest in Civil Engineering**
> A neural network might give you a slightly higher accuracy, but it cannot tell you *which features mattered*. Feature importance is the bridge from "the model says X" to "here is the engineering insight that defends the recommendation." For any policy or retrofit decision, that bridge is essential.

**Caveats** (we'll revisit in Exercise 4):
- High-cardinality features (like `geo_level_3_id` with thousands of levels) tend to score artificially high.
- Strongly correlated features split credit between themselves — neither looks as important as it really is.

In [ ]:
def plot_feature_importances(model, feature_names, top_n=15, ax=None,
                             color='steelblue', title='Feature importances'):
    """Horizontal bar chart of the top-n features by .feature_importances_."""
    importances = model.feature_importances_
    idx = np.argsort(importances)[-top_n:]
    if ax is None:
        ax = plt.gca()
    colors = [color] * len(idx)
    # Highlight the top 3 in indianred
    for i in range(1, 4):
        if i <= len(colors):
            colors[-i] = 'indianred'
    ax.barh(np.array(feature_names)[idx], importances[idx], color=colors)
    ax.set_xlabel('Importance (mean Gini reduction across forest)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='x')
    return ax

# Recover feature names after the ColumnTransformer
feature_names = cat_cols + num_cols   # OrdinalEncoder doesn't rename
rf_clf = rf_pipe.named_steps['clf']

fig, ax = plt.subplots(figsize=(10, 8))
plot_feature_importances(rf_clf, feature_names, top_n=15, ax=ax,
                         title='Top 15 features predicting Nepal earthquake damage')
plt.tight_layout()
plt.show()

### The Nepal Vulnerability Story

Read the plot top-down. Three things stand out — and one of them is a trap.

1. **Location dominates — by a wide margin.** All three `geo_level_*_id` features sit at the top of the chart, together accounting for roughly 40% of the model's total impurity reduction. This is not surprising: the Gorkha quake's shaking intensity varied sharply with distance from the rupture, and a building's ward also encodes the local construction tradition (which materials are common, how skilled the masons are, whether retrofits have been done).

2. **Age and geometry come next.** `age`, `area_percentage`, and `height_percentage` together explain another ~25%. Older, larger buildings are at higher risk — again, exactly what an experienced engineer would have guessed.

3. **Construction-type features rank surprisingly low — but this is misleading.** `foundation_type` and `roof_type` show up well below the geographic features, and *none* of the eleven `has_superstructure_*` flags appear in the top 15 at all. Does that really mean materials don't matter? **No.** It means we are looking at *Gini* importance, which has two well-known biases:
   - **High-cardinality features get inflated.** `geo_level_3_id` has thousands of distinct values, giving it many more split opportunities than a binary flag. Its 0.15 importance is partly an artefact of the encoding, not a pure signal.
   - **Correlated features split credit.** The eleven `has_superstructure_*` flags are highly correlated (a mud-mortar-stone building rarely also has reinforced concrete). Each flag absorbs only a small share of the importance the underlying material concept actually carries.

> **Key Insight: Always Sanity-Check Importance**
> The first feature-importance plot you make is *the start of the conversation*, not the end. **Exercise 4** asks you to recompute these with `permutation_importance` — which doesn't care about cardinality and is much harder to fool. You'll see the ranking shift, and the construction-material story emerge more clearly.

4. **The honest policy lesson.** What this plot *does* tell us, even with all the caveats: **where a building stands matters at least as much as how it was built.** If a regional government had a finite retrofit budget, the first cut would be by district — focus on the hardest-hit wards, then within those wards address material vulnerabilities. **This is the entire reason to choose Random Forest over a black-box model — every recommendation comes with a feature-importance audit trail you can defend.** It is also exactly the kind of analysis you should be aiming to produce for your final project.

---

## 6. Regression, Summary & Practice

### Trees Predict Numbers Too

Everything we've done so far is *classification* — predicting a discrete label. Trees handle *regression* (predicting a continuous number) with almost no change to the algorithm:

- The split criterion changes from **Gini** to **MSE** (mean squared error of the residuals).
- Each leaf predicts the **mean** of the training samples that landed in it.
- Bagging, random feature subsetting, feature importance, the overfitting story — all identical.

A quick visual demo on a synthetic 1-D regression problem makes the difference between a single tree and a forest crystal clear.

In [ ]:
X_reg, y_reg = make_regression(n_samples=300, n_features=1, noise=15, random_state=49)
X_reg = X_reg.ravel()
sort_idx = np.argsort(X_reg)
X_reg, y_reg = X_reg[sort_idx], y_reg[sort_idx]

dt_reg = DecisionTreeRegressor(max_depth=3, random_state=49).fit(X_reg.reshape(-1, 1), y_reg)
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=49,
                               n_jobs=-1).fit(X_reg.reshape(-1, 1), y_reg)

print(f"Single tree (depth 3) R^2 : {dt_reg.score(X_reg.reshape(-1, 1), y_reg):.3f}")
print(f"Random Forest (100 trees) R^2 : {rf_reg.score(X_reg.reshape(-1, 1), y_reg):.3f}")

In [ ]:
x_grid = np.linspace(X_reg.min() - 0.5, X_reg.max() + 0.5, 500).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_reg, y_reg, color='lightgray', s=30, label='Training data')
ax.plot(x_grid, dt_reg.predict(x_grid), color='steelblue', linewidth=2.5,
        label='Single tree (depth=3)')
ax.plot(x_grid, rf_reg.predict(x_grid), color='indianred', linewidth=2.5,
        label='Random Forest (100 trees)')
ax.set_xlabel('Feature')
ax.set_ylabel('Target')
ax.set_title('Tree regression: staircase of one tree vs. smoothed average of a forest')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

> **Key Insight: Same Algorithm, Two Jobs**
> Classification trees split to reduce *Gini*; regression trees split to reduce *MSE*. Everything else — bagging, feature importance, the overfitting story — works identically. If you've understood today's classification material, you've understood regression trees too.

### Decision Tree & Random Forest Advantages

| Advantage | Why it matters in CE work |
|---|---|
| **No scaling needed** | Mix `age_years` and `area_m2` in the same model without a `StandardScaler`. |
| **Mixed data types** | Categorical `foundation_type` next to numeric `count_floors` — no fuss. |
| **Single tree is interpretable** | An inspector can audit the if-then rules. |
| **Reliable feature importance** | The headline output for any CE policy / retrofit study. |
| **Captures non-linear interactions for free** | No kernel, no feature engineering. |
| **Few hyperparameters that matter** | `max_depth`, `min_samples_leaf`, `n_estimators` — that's it. |
| **Robust to outliers** | Splits care about ordering, not magnitudes. |

### Disadvantages

| Disadvantage | Mitigation |
|---|---|
| Single trees overfit easily | Use a Random Forest. |
| Boundaries are axis-aligned (poor for diagonal patterns) | RF with many trees smooths this out. SVM with RBF if it matters. |
| Slight changes in data → very different tree | Ensembling fixes this. |
| RF loses interpretability of a *single* prediction | Keep the feature-importance plot for global insight. |
| Slow to predict if the forest is huge | `n_estimators ≤ 500` is usually plenty. |
| Gini importance is biased toward high-cardinality features | Use `permutation_importance` for a sanity check (Exercise 4). |

### When to Choose Trees vs. Other Models

**Reach for trees / RF when:**
- Your data is **tabular** with mixed feature types (the dominant CE case).
- You need **feature importance** to defend a recommendation.
- You don't have time to scale, encode, and tune carefully.
- You want a strong **baseline** before trying anything fancier.

**Use something else when:**
- Sparse, high-dimensional **text** data → Naive Bayes.
- Clean numeric data with **diagonal/curved** structure → SVM with RBF.
- You need calibrated **probabilities** → logistic regression or gradient boosting + calibration.
- The dataset is **tiny** (<100 rows) → simpler parametric models.

> **Practical Recommendation**
> For tabular civil-engineering data: **always try a Random Forest first.** It will be your baseline, and often your final model. Move to gradient-boosted trees (XGBoost, LightGBM) if you need a few more accuracy points and have time to tune.

### Key Takeaways

1. A decision tree is a **flowchart of yes/no questions** that ends in a prediction.
2. **Gini impurity** measures how "mixed" a node is; the tree greedily picks the split that drops weighted Gini the most.
3. Unrestricted trees **memorize the training set** — prune with `max_depth` and `min_samples_leaf`, guided by a **validation curve**.
4. **Random Forests** = bagging (bootstrap rows) + random feature subsets at each split. The two tricks decorrelate the trees so that averaging them really helps.
5. **`feature_importances_`** is your interpretability superpower — and the bridge from prediction to engineering insight.
6. Trees **don't need scaling** and handle **mixed feature types** natively — perfect for messy CE tabular data.
7. **Random Forest is your default baseline** for any tabular CE problem. Always try it first.

**The tree workflow:**
1. Light preprocessing (encode categoricals with `OrdinalEncoder`).
2. Baseline `DecisionTreeClassifier` with a moderate `max_depth`.
3. Jump to `RandomForestClassifier(n_estimators=200, n_jobs=-1)`.
4. Inspect `feature_importances_` — read the engineering story.
5. Tune `max_depth` / `min_samples_leaf` only if needed.

---

## [PRACTICE] Practice Exercises

Try these on your own. Exercises 4 and 5 are excellent warm-ups for your Week-10-12 final project.

1. **Exercise 1 (Warm-up — interpretability):** Load the iris dataset (`from sklearn.datasets import load_iris`). Train a `DecisionTreeClassifier(max_depth=3)` and visualize it with `plot_tree(...)`. In a markdown cell, write 2-3 sentences explaining the if-then rules the tree learned. Which feature does the root split on, and why does that make sense?

2. **Exercise 2 (The depth dial):** Using the Nepal dataset, train decision trees with `max_depth ∈ {1, 3, 5, 10, 20, None}`. Plot train and cross-validated test accuracy vs. depth (you can reuse `plot_validation_curve`). Identify the depth where overfitting clearly begins. **Bonus:** also sweep `min_samples_leaf ∈ {1, 5, 20, 100}` — which knob gives a smoother curve?

3. **Exercise 3 (Forest size):** On the Nepal dataset, train random forests with `n_estimators ∈ {1, 5, 25, 100, 500}`. Plot test accuracy and training time on the same axes (use `ax.twinx()` for the second y-axis). At what point do you hit diminishing returns? Justify the `n_estimators` you would actually use in production.

4. **Exercise 4 (Permutation importance vs Gini importance):** On the Nepal RF, also compute `sklearn.inspection.permutation_importance` and plot the top 10. Compare against the built-in `feature_importances_`. Which features change rank? Write a paragraph explaining why permutation importance is sometimes more trustworthy (hint: think about high-cardinality features like `geo_level_3_id`).

5. **Exercise 5 (Final-project warm-up):** Pick *any* tabular dataset relevant to your final-project domain — a CSV from your team's data source (geotechnical, structural, traffic, hydrology, energy, building energy, sensor data, …). Build a `RandomForestClassifier` *or* `RandomForestRegressor` pipeline end-to-end: encode categoricals, split, fit, score, plot the top-10 feature importances. Submit:
   - (a) the test metric (accuracy or R²),
   - (b) the top-10 feature-importance figure,
   - (c) **one paragraph** of engineering interpretation: which features matter, are the rankings physically plausible, and what would your project team actually *do* with this information?

   *Treat this as a dry run for the final project.*

In [ ]:
# Your code here

---

### Questions?

**Dr. Eyuphan Koc**
eyuphan.koc@bogazici.edu.tr